In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Atténuation des erreurs par réseau de tenseurs (TEM) : une fonction Qiskit par Algorithmiq
> **Note:** Les fonctions Qiskit sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les prérequis suivants.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Vue d'ensemble
La méthode d'atténuation des erreurs par réseau de tenseurs (TEM) d'Algorithmiq est un algorithme hybride quantique-classique conçu pour effectuer l'atténuation du bruit entièrement lors de l'étape de post-traitement classique. Grâce à TEM, tu peux calculer les valeurs d'espérance des observables en atténuant les inévitables erreurs induites par le bruit qui surviennent sur le matériel quantique, avec une précision accrue et une meilleure rentabilité — ce qui en fait une option très attrayante pour les chercheurs en informatique quantique et les praticiens industriels.

La méthode consiste à construire un réseau de tenseurs représentant l'inverse du canal de bruit global affectant l'état du processeur quantique, puis à appliquer cette transformation aux résultats de mesure informationnellement complets acquis depuis l'état bruité afin d'obtenir des estimateurs non biaisés pour les observables.

L'un des atouts de TEM est qu'il tire parti de mesures informationnellement complètes pour donner accès à un vaste ensemble de valeurs d'espérance atténuées des observables, tout en ayant un surcoût d'échantillonnage optimal sur le matériel quantique, comme décrit dans Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), et Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Le surcoût de mesure désigne le nombre de mesures supplémentaires nécessaires pour réaliser une atténuation d'erreur efficace — un facteur déterminant pour la faisabilité des calculs quantiques. TEM a donc le potentiel d'ouvrir la voie à l'avantage quantique dans des scénarios complexes, comme les applications en chaos quantique, en physique des systèmes à plusieurs corps, en dynamique de Hubbard et en simulation de chimie de petites molécules.

Les principales caractéristiques et avantages de TEM peuvent être résumés comme suit :

1.  **Surcoût de mesure optimal** : TEM est optimal par rapport aux
[bornes théoriques](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
ce qui signifie qu'aucune méthode ne peut atteindre un surcoût de mesure inférieur. Autrement dit, TEM requiert le nombre minimal de mesures supplémentaires pour effectuer l'atténuation des erreurs, ce qui se traduit par une utilisation minimale du temps d'exécution quantique.
2.  **Rentabilité** : Comme TEM gère entièrement l'atténuation du bruit lors de la phase de post-traitement, il n'est pas nécessaire d'ajouter des circuits supplémentaires sur l'ordinateur quantique — ce qui rend le calcul moins coûteux et diminue le risque d'introduire des erreurs supplémentaires dues aux imperfections des appareils quantiques.
3.  **Estimation de plusieurs observables** : Grâce aux mesures informationnellement complètes, TEM estime efficacement plusieurs observables à partir des mêmes données de mesure provenant de l'ordinateur quantique.
4.  **Atténuation des erreurs de mesure** : La fonction Qiskit TEM inclut également une
[méthode propriétaire d'atténuation des erreurs de mesure](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
capable de réduire significativement les erreurs de lecture après une courte exécution de calibration.
5.  **Précision** : TEM améliore considérablement la précision et la fiabilité des simulations quantiques numériques, rendant les algorithmes quantiques plus exacts et plus robustes.
## Description
La fonction TEM te permet d'obtenir des valeurs d'espérance atténuées pour plusieurs observables sur un circuit quantique avec un surcoût d'échantillonnage minimal. Le circuit est mesuré avec une mesure à opérateur à valeur positive informationnellement complète (IC-POVM), et les résultats de mesure collectés sont traités sur un ordinateur classique. Cette mesure est utilisée pour appliquer les méthodes de réseau de tenseurs et construire une carte d'inversion du bruit. La fonction applique une transformation qui inverse entièrement le circuit bruité en utilisant des réseaux de tenseurs pour représenter les couches bruyantes.

![Schémas TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimation atténuée en erreur d'une observable O via le post-traitement des résultats de mesure du processeur quantique bruité. U et N désignent respectivement une opération quantique idéale et la carte de bruit associée, qui peut être généralement non locale (et étendue aux boîtes grises). D représente un tenseur d'opérateurs duaux aux effets de la mesure IC. Le module d'atténuation du bruit M est un réseau de tenseurs contracté efficacement de l'intérieur vers l'extérieur. La première itération de la contraction est représentée par la ligne pointillée violette, la deuxième par la ligne tiretée, et la troisième par la ligne continue.")

Une fois les circuits soumis à la fonction, ils sont transpilés et optimisés afin de minimiser le nombre de couches comportant des portes à deux qubits (les portes les plus bruyantes sur les appareils quantiques). Le bruit affectant les couches est appris via
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
en utilisant un modèle de bruit de Pauli-Lindblad sparse tel que décrit dans E. van den Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Le modèle de bruit est une description précise du bruit sur l'appareil, capable de capturer des caractéristiques subtiles, notamment la diaphonie entre qubits. Cependant, le bruit sur les appareils peut fluctuer et dériver, et le bruit appris peut ne pas être exact au moment où l'estimation est effectuée. Cela peut entraîner des résultats imprécis.
## Premiers pas
Authentifie-toi avec ta [clé API IBM Quantum Platform](http://quantum.cloud.ibm.com/) et sélectionne la fonction TEM comme suit. (Cet extrait suppose que tu as déjà [sauvegardé ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Exemple
L'extrait suivant montre un exemple où TEM est utilisé pour calculer les valeurs d'espérance d'une observable pour un circuit quantique simple.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Utilise les API Qiskit Serverless pour vérifier le statut de ton workload Qiskit Function :

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Tu peux récupérer les résultats comme suit :

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** La valeur d'espérance pour le circuit sans bruit pour l'opérateur donné devrait être d'environ `0.18409094298943401`.
## Entrées
**Paramètres**

Nom | Type | Description | Requis | Par défaut | Exemple
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Un itérable d'objets de type PUB (primitive unified bloc), tels que des tuples `(circuit, observables)` ou `(circuit, observables, parameters, precision)`. Voir [Vue d'ensemble des PUBs](/guides/primitive-input-output#overview-of-pubs) pour plus d'informations. Si un circuit non-ISA est passé, il sera transpilé avec des paramètres optimaux. Si un circuit ISA est passé, il ne sera pas transpilé ; dans ce cas, l'observable doit être définie sur l'ensemble du QPU. | Oui | N/A | (circuit, observables)
`backend_name` | str | Nom du backend à utiliser pour la requête. | Non | Si non fourni, le backend le moins occupé sera utilisé. | "ibm_fez"
`options` | dict | Options d'entrée. Voir la section `Options` pour plus de détails. | Non | Voir la section `Options` pour plus de détails. | {"max_bond_dimension": 100\}

> **Caution:** TEM présente actuellement les limitations suivantes :
> 
>   - Les circuits paramétrés ne sont pas pris en charge. L'argument parameters doit être défini à `None` si la précision est spécifiée. Cette restriction sera supprimée dans les versions futures.
>   - Seuls les circuits sans boucles sont pris en charge. Cette restriction sera supprimée dans les versions futures.
>   - Les portes non unitaires, telles que reset, measure, et toutes les formes de flux de contrôle ne sont pas prises en charge. La prise en charge de reset sera ajoutée dans les prochaines versions.
### Options
Un dictionnaire contenant les options avancées pour TEM. Le dictionnaire peut contenir les clés du tableau suivant. Si certaines options ne sont pas fournies, la valeur par défaut indiquée dans le tableau sera utilisée. Les valeurs par défaut conviennent à une utilisation typique de TEM.

Nom | Choix | Description | Par défaut
-- | -- | -- | --
`tem_max_bond_dimension` | int | La dimension de liaison maximale à utiliser pour les réseaux de tenseurs. | 500 |
`tem_compression_cutoff` | float | La valeur de coupure à utiliser pour les réseaux de tenseurs. | 1e-16
`compute_shadows_bias_from_observable` | bool | Un indicateur booléen indiquant si le biais pour le protocole de mesure des ombres classiques doit être adapté à l'observable du PUB ou non. Si False, le protocole des ombres classiques (probabilité égale de mesurer Z, X, Y) sera utilisé. | False
`shadows_bias` | np.ndarray | Le biais à utiliser pour le protocole de mesure des ombres classiques aléatoires, un tableau 1d ou 2d de taille 3 ou de forme (num_qubits, 3) respectivement. L'ordre est ZXY. | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int ou `None` | Le temps d'exécution maximal sur le QPU en secondes. Si la durée d'exécution dépasse cette valeur, le job sera annulé. Si `None`, une limite par défaut définie par Qiskit Runtime s'appliquera. | `None`
`num_randomizations` | int | Le nombre de randomisations à utiliser pour l'apprentissage du bruit et le twirling des portes. | 32
`max_layers_to_learn` | int | Le nombre maximum de couches uniques à apprendre. | 4
`mitigate_readout_error` | bool | Un indicateur booléen indiquant si l'atténuation des erreurs de lecture doit être effectuée ou non. | True
`num_readout_calibration_shots` | int | Le nombre de shots à utiliser pour l'atténuation des erreurs de lecture. | 10000
`default_precision` | float | La précision par défaut à utiliser pour les PUBs pour lesquels la précision n'est pas spécifiée. | 0.02
`seed` | int ou `None` | Définit la graine du générateur de nombres aléatoires pour la reproductibilité. Si `None`, la graine n'est pas définie. | None
## Sorties
Un objet Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) contenant le résultat atténué par TEM. Le résultat pour chaque PUB est retourné sous forme de [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) contenant les champs suivants :

Nom | Type | Description
-- | -- | --
data | DataBin | Un objet Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) contenant l'observable atténuée par TEM et son erreur standard. Le DataBin possède les champs suivants : <ul><li>`evs` : La valeur de l'observable atténuée par TEM.</li><li>`stds` : L'erreur standard de l'observable atténuée par TEM.</li></ul>
metadata | dict | Un dictionnaire contenant des résultats supplémentaires. Le dictionnaire contient les clés suivantes : <ul><li>`"evs_non_mitigated"` : La valeur de l'observable sans atténuation des erreurs.</li><li>`"stds_non_mitigated"` : L'erreur standard du résultat sans atténuation des erreurs.</li><li>`"evs_mitigated_no_readout_mitigation"` : La valeur de l'observable avec atténuation des erreurs mais sans atténuation des erreurs de lecture.</li><li>`"stds_mitigated_no_readout_mitigation"` : L'erreur standard du résultat avec atténuation des erreurs mais sans atténuation des erreurs de lecture.</li><li>`"evs_non_mitigated_with_readout_mitigation"` : La valeur de l'observable sans atténuation des erreurs mais avec atténuation des erreurs de lecture.</li><li>`"stds_non_mitigated_with_readout_mitigation"` : L'erreur standard du résultat sans atténuation des erreurs mais avec atténuation des erreurs de lecture.</li></ul>
## Récupérer les messages d'erreur
Si le statut de ton workload est ERROR, utilise `job.result()` pour récupérer le message d'erreur comme suit :